# v05 · HyperGallery: images and cubes as addresses; the JSON master index
`hypergallery.py` (Data-Storage copy).

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
hg = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hypergallery.py", "hg")
def roundtrip(spec, vals, tag):
    G = hg.HyperGallery(spec); rec = G.index_image(vals, tag)
    assert G.regenerate_image(rec) == vals
    print("%-22s n_values=%7d  address words=%5d  bits(spec)=%d" % (tag, spec.n_values, len(rec.label.split("_")), spec.address_bits))
    return rec
r1 = roundtrip(hg.ImageSpec(106, 17, hg.PixelMode.BINARY), [(i + j) % 2 for j in range(17) for i in range(106)], "binary 106x17")
r2 = roundtrip(hg.ImageSpec(64, 64, hg.PixelMode.GRAY8), [(i * 255 // 4096) for i in range(4096)], "gray8 64x64")
r3 = roundtrip(hg.ImageSpec(16, 16, hg.PixelMode.SPECTRAL, n_bands=50, bit_depth=16), [(x + y + k * 7) % 65536 for y in range(16) for x in range(16) for k in range(50)], "IFU 16x16x50x16bit")

binary 106x17          n_values=   1802  address words=   57  bits(spec)=1802
gray8 64x64            n_values=   4096  address words= 1024  bits(spec)=32768
IFU 16x16x50x16bit     n_values=  12800  address words= 6400  bits(spec)=204800


The address of an image with n values at depth D is a base-D numeral with n digits: `n·log₂D` bits. That is the size of the image itself; nothing is saved. The address exists whether or not the image was ever recorded.

## The JSON master index: many records → one address

In [3]:
idx = hg.HyperIndex()
for r in (r1, r2, r3): idx.add_image_record(r)
js = idx.to_json(); m = idx.master_address()
back = idx.regenerate_from_master(m)
assert len(back["image_records"]) == 3
print("index JSON:", len(js), "chars ->", m.num_words, "uint32 words =", m.num_words*32, "bits;", "ratio to raw JSON bits:", round(m.num_words*32/(8*len(js)), 3))

index JSON: 68161 chars -> 14090 uint32 words = 450880 bits; ratio to raw JSON bits: 0.827
